In [ ]:
%matplotlib inline
import numpy as np
import matplotlib.pyplot as plt

# 99.4 Optimization Basics

## Contents

**Tools:** CADET-Process (Python).

**Python used:** Functions, lists, bounds and optimizer configuration.

**Introduces:** `OptimizationProblem`, objectives, `TrustConstr`, linear and nonlinear constraints.

1. Define an objective and bounds
2. Run an optimizer and read the result
3. Add constraints

Optimization chooses parameter values that improve a criterion. In chromatography, that might mean increasing recovery or reducing the difference between a simulation and a measurement.

Here, a simple function makes every step visible before we connect optimization to a process.

```{note}
For this small mathematical example, a direct calculation or a SciPy optimizer is sufficient. We use it to demonstrate the CADET-Process interface: the same variables, objectives and constraints can be passed to different algorithms. Its evaluation chains and caching become useful when each candidate requires simulations, comparisons or fractionation.
```

## Define the problem

For $f(x)=x^2$, the minimum is at $x=0$. Restrict the search to **−10 ≤ x ≤ 10**.

`OptimizationProblem` holds the variables, objectives and constraints. The optimizer decides which values to try.

In [ ]:
from CADETProcess.optimization import OptimizationProblem

quadratic_problem = OptimizationProblem("quadratic")

In [ ]:
quadratic_problem.add_variable("x", lb=-10, ub=10)

The objective receives a list of variable values, in the order they were added. `x[0]` is the first value; the function returns the criterion to minimize.

In [ ]:
def quadratic(x):
    return x[0] ** 2

quadratic_problem.add_objective(quadratic, name="squared_value")

Before starting an optimizer, we evaluate one candidate. At **x = 2**, the objective should be **4**.

In [ ]:
print("Objective at x = 2:", quadratic_problem.evaluate_objectives([2]))

## Run the optimizer

`TrustConstr` is a local optimizer: it starts at `x0` and searches nearby for improvement. Here, start at **x = 2**. `n_max_iter` limits the number of iterations.

Supply `x0` in the order the variables were added, in their physical units and within the bounds. For a constrained problem, choose a feasible start where possible. Different starts can lead to different local minima; Exercise 2 compares this directly with a population search.

In [ ]:
from CADETProcess.optimization import TrustConstr

quadratic_optimizer = TrustConstr()
quadratic_optimizer.n_max_iter = 50
quadratic_results = quadratic_optimizer.optimize(quadratic_problem, x0=[2])
print("Best x:", quadratic_results.x)
print("Objective:", quadratic_results.f)

## Read the result

`.x` contains the best parameter values; `.f` contains their objective values. Both are arrays because a problem can have several variables, objectives or equally good solutions.

The convergence plot shows whether the objective improves as candidates are evaluated.

In [ ]:
_ = quadratic_results.plot_convergence("objectives")
plt.show()

```{tip}
Optimizers also save results and figures in their results directory. `results_all.csv` contains evaluated candidates; `results_pareto.csv` contains the selected best candidates.
```

## Add constraints

Bounds restrict each variable separately. Constraints restrict combinations of values.

For two variables, minimize **$x^2+y^2$** with **$x+y=1$** and **$x\leq0.4$**, within **0 ≤ x, y ≤ 1**. The best feasible point is **[0.4, 0.6]**.

In [ ]:
constrained_problem = OptimizationProblem("constrained_quadratic")
constrained_problem.add_variable("x", lb=0, ub=1)
constrained_problem.add_variable("y", lb=0, ub=1)

def two_variable_objective(values):
    return values[0] ** 2 + values[1] ** 2

constrained_problem.add_objective(two_variable_objective)

For a linear constraint, `lhs` gives the coefficients in the listed variable order. Inequalities use **≤ b**; equalities use **= b**.

In [ ]:
constrained_problem.add_linear_equality_constraint(["x", "y"], lhs=[1, 1], b=1)
constrained_problem.add_linear_constraint(["x"], lhs=[1], b=0.4)

A nonlinear constraint takes a function. This additional condition keeps **$x^2+y^2\leq0.8$**. State the comparison explicitly with `comparison_operator="le"` (less than or equal).

In [ ]:
constrained_problem.add_nonlinear_constraint(
    two_variable_objective, name="squared_limit", bounds=0.8,
    comparison_operator="le",
)

In [ ]:
start = [0.3, 0.7]
print("Linear inequality satisfied:", constrained_problem.check_linear_constraints(start))
print("Linear equality satisfied:", constrained_problem.check_linear_equality_constraints(start))
print("Nonlinear constraint satisfied:", constrained_problem.check_nonlinear_constraints(start))

## Optimize within the feasible region

The starting values satisfy all constraints. The optimizer now improves the objective while respecting those restrictions.

In [ ]:
constrained_optimizer = TrustConstr()
constrained_optimizer.n_max_iter = 100
constrained_results = constrained_optimizer.optimize(constrained_problem, x0=start)
print("Best [x, y]:", constrained_results.x)
print("Objective:", constrained_results.f)
_ = constrained_results.plot_convergence("objectives")
plt.show()

```{note}
A solver uses numerical tolerances, so the result may be close to a bound rather than exactly on it. Interpreting a result therefore means looking at both the objective and constraint satisfaction.
```

## Summary

- `OptimizationProblem` stores variables, objectives and constraints.
- Starting values and solver choice affect the search.
- Objective and constraint values of the selected point hold only within numerical tolerances.

**Now:** **99.4 Optimization Basics: Exercises**: compare local and population searches.

**Next lesson:** **99.5 Advanced Optimization** connects variables to simulated processes.